# RL Run Analysis

This notebook analyzes W&B runs from RL training experiments (using `hf_trainer` with GRPOTrainer).
Fetches and visualizes reward metrics, parsing statistics, training progress, and sample outputs.

**Setup:** Ensure wandb is configured (`wandb login` or API key in `.env`).

In [ ]:
import html
import json
import typing

import IPython.display as ipy_display
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pyine.utils.filesystem
import pyine.utils.notebooks as nb_utils
import pyine.utils.reprod
import pyine.utils.wandb_utils as wandb_utils

In [ ]:
pyine.utils.reprod.entrypoint_setup()
nb_utils.setup_notebook_plotting(use_seaborn=True, seaborn_style="whitegrid")

# ------------ CONFIGURATION ------------
WANDB_PROJECT = "pyine-tests"  # change to your project
WANDB_ENTITY = None  # optional: your team/user entity

# option 1: specify run directly by URL or ID (takes priority)
WANDB_RUN_URL = ""  # e.g., "https://wandb.ai/entity/project/runs/abc123"
WANDB_RUN_ID = ""  # e.g., "abc123"

# option 2: search for runs using filters (used when URL/ID not specified)
# W&B API filters (applied server-side) - use MongoDB-style query syntax
RUN_FILTERS: dict[str, typing.Any] = {
    # RL runs have num_generations > 0 (GRPO generates multiple completions per prompt)
    "config.num_generations": {"$gt": 0},
    # "state": "finished",  # uncomment to only show completed runs
    # "group": "...",  # uncomment to filter by run group
}
# optional client-side filter (set to None to skip); applied after server-side filters
RUN_FILTER_FN: typing.Callable[[typing.Any], bool] | None = None
SELECTED_RUN_IDX = 0  # which run to select from search results (0 = most recent)

# optional: parquet cache path for large runs (set to None to disable)
history_cache_dir = pyine.utils.filesystem.get_data_cache_subdir("notebooks", "cached_wandb_runs")

In [ ]:
# load run: try direct URL/ID first, then fall back to filter-based search
if WANDB_RUN_URL or WANDB_RUN_ID:
    # option 1: direct run specification
    run = wandb_utils.get_wandb_run(WANDB_RUN_URL, WANDB_RUN_ID, WANDB_PROJECT, WANDB_ENTITY)
    print(f"Loaded run directly: {run.name} ({run.url})")
else:
    # option 2: search for runs using filters
    print(f"Searching for runs with filters: {RUN_FILTERS}")
    runs = wandb_utils.fetch_runs(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        filters=RUN_FILTERS if RUN_FILTERS else None,
        order="-created_at",  # newest first
        per_page=50,
    )
    # apply client-side filter if provided (e.g., to filter for RL runs)
    if RUN_FILTER_FN is not None:
        runs = [r for r in runs if RUN_FILTER_FN(r)]
        print(f"After client-side filter: {len(runs)} RL runs")
    if not runs:
        raise ValueError(f"No runs found matching filters (server: {RUN_FILTERS}, client: {RUN_FILTER_FN})")
    print(f"Found {len(runs)} matching runs:")
    for idx, r in enumerate(runs[:10]):  # show up to 10
        marker = " <-- selected" if idx == SELECTED_RUN_IDX else ""
        print(f"  [{idx}] {r.group}/{r.name} ({r.state}) - {r.created_at}{marker}")
    if len(runs) > 10:
        print(f"  ... and {len(runs) - 10} more")
    if len(runs) <= SELECTED_RUN_IDX:
        raise ValueError(f"SELECTED_RUN_IDX={SELECTED_RUN_IDX} but only {len(runs)} runs found")
    run = runs[SELECTED_RUN_IDX]
    print(f"\nSelected run: {run.name} ({run.url})")

print(f"State: {run.state}, Created: {run.created_at}")

In [ ]:
# display key summary metrics and configuration
summary = dict(run.summary)
config = dict(run.config).get("main_config", {})


def get_nested_config_value(
    config: dict[str, typing.Any],
    dotted_key: str,
) -> typing.Any | None:
    """Retrieve a value from a nested config dict using dot notation (e.g., 'grpo_config.beta')."""
    parts = dotted_key.split(".")
    current = config
    for part in parts:
        if isinstance(current, dict) and part in current:
            current = current[part]
        else:
            return None
    return current


print(f"Run Configuration: {run.name} ({run.url})")
config_keys_of_interest = [
    "base_model",
    "datamodule_config.lmdb_paths",
    "grpo_config.per_device_train_batch_size",
    "grpo_config.per_device_eval_batch_size",
    "grpo_config.gradient_accumulation_steps",
    "grpo_config.learning_rate",
    "grpo_config.weight_decay",
    "grpo_config.gradient_checkpointing",
    "grpo_config.num_train_epochs",
    "grpo_config.num_generations",
    "grpo_config.max_completion_length",
    "grpo_config.max_prompt_length",
    "grpo_config.temperature",
    "grpo_config.beta",
]
for key in config_keys_of_interest:
    value = get_nested_config_value(config, key)
    if value is not None:
        if isinstance(value, (tuple, list)):
            print(f"  {key}:")
            for item in value:
                print(f"\t- {repr(item)}")
        elif isinstance(value, dict):
            for k, v in value.items():
                print(f"  {key}.{k}: {repr(v)}")
        else:
            print(f"  {key}: {repr(value)}")

print("\nRun Summary (key metrics):")
summary_keys_of_interest = [
    "eval/reward/run/categories/tags/difficulty/EASY/mean",
    "eval/reward/run/categories/tags/difficulty/MEDIUM/mean",
    "eval/reward/run/categories/tags/difficulty/MEDIUM_HARD/mean",
    "eval/reward/run/categories/tags/difficulty/HARD/mean",
    "eval/reward/run/categories/tags/difficulty/VERY_HARD/mean",
    "eval/reward/run/categories/tags/difficulty/UNKNOWN_DIFFICULTY/mean",
    "eval/rewards/TRLRewardAdapter/mean",
    "eval/reward/run/total/mean",  # check: should be the same as eval/rewards/TRLRewardAdapter/mean
    "eval/reward/run/total/count",
    "eval/parsing/missing_answer_ratio",
    "eval/samples_per_second",
    "eval/failures/failure_count",
    "train/failures/failure_count",
    "train/reward/run/total/mean",
    "train/reward/run/total/count",
    "train/parsing/missing_answer_ratio",
    "train/generation_count",
    "train/global_step",
    "train/epoch",
]
for key in summary_keys_of_interest:
    if key in summary:
        print(f"  {key}: {summary[key]}")

In [ ]:
available_keys = wandb_utils.discover_metric_keys(run)

print("Available Metrics in This Run:")
for category, keys in available_keys.items():
    if keys:
        print(f"\n  {category} ({len(keys)} keys):")
        for key in keys:
            print(f"    - {key}")
    else:
        print(f"\n  {category}: (none)")

In [ ]:
# fetch important metrics (focus mostly on run-wise logs, not sporadic logs)
important_key_parts = [
    "/reward/run/total/",
    "/reward/run/terms/",
    "/reward/metrics/verbosity/factor/clip_ratio/",
]
important_key_suffixes = [
    "/failures/failure_count",
    "/parsing/missing_reasoning_ratio",
    "/parsing/missing_answer_ratio",
    "/parsing/output_length_chars/mean",
    "/parsing/output_length_tokens/mean",
    "/parsing/reasoning_length_chars/mean",
    "/parsing/reasoning_length_tokens/mean",
    "/entropy",
    "/kl",
    "/step_time",
    "/num_tokens",
    "/samples_per_second",
    "/steps_per_second",
]
available_keys_flat = [key for keys in available_keys.values() for key in keys]
important_keys = sorted(
    set(
        [
            key
            for key in available_keys_flat
            if any(s in key for s in important_key_parts) or any(key.endswith(s) for s in important_key_suffixes)
        ]
        + available_keys["step_keys"]  # always grab all step keys as important
    )
)

# only cache history for finished runs (incomplete runs would have stale cached data)
history_cache_path = history_cache_dir / f"{run.id}.parquet" if run.state != "running" else None
if history_cache_path is None:
    print(f"Run state is '{run.state}' - caching disabled (will fetch fresh data)")
history_df = wandb_utils.fetch_history_df(
    run,
    keys=important_keys if important_keys else None,
    cache_path=history_cache_path,
)
step_key = wandb_utils.resolve_step_key(history_df)
time_key = wandb_utils.resolve_time_key(history_df)

print(f"\nFetched {len(history_df)} history rows with {len(history_df.columns)} columns")
print(f"Using step key: {step_key}")
if time_key:
    print(f"Time key available: {time_key}")

In [ ]:
important_keys
# history_cache_path

In [ ]:
history_df  # noqa: B018 (for display purposes)

## Reward Metrics

In [ ]:
# color palette for multi-line plots (colorblind-friendly)
TERM_COLORS = ["#2C7BB6", "#D7191C", "#1A9641", "#FDAE61", "#9970AB", "#E7298A", "#66A61E", "#E6AB02"]


def _get_std_key_for_mean_key(mean_key: str) -> str | None:
    """Derive the std key from a mean key by replacing /mean suffix with /std."""
    if mean_key.endswith("/mean"):
        return mean_key[:-5] + "/std"
    return None


def _get_count_key_for_mean_key(mean_key: str) -> str | None:
    """Derive the count key from a mean key."""
    if mean_key.endswith("/mean"):
        return mean_key[:-5] + "/count"
    return None


def plot_reward_over_time(
    history_df: pd.DataFrame,
    mean_keys: list[str] | str = ("train/reward/run/total/mean", "eval/reward/run/total/mean"),
    step_key: str = "train/global_step",
    ax: plt.Axes | None = None,
    figsize: tuple[int, int] = (12, 5),
    title: str | None = None,
    show_scatter: bool = True,
    show_legend: bool = True,
    show_ci: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
    colors: list[str] | None = None,
) -> matplotlib.figure.Figure:
    """Plot reward metrics over training steps using logged mean/std values.

    Args:
        history_df: DataFrame with history data.
        mean_keys: Name(s) of the mean reward column(s) to plot. Can be a single string or list.
        step_key: Name of the step column (x-axis).
        ax: Optional matplotlib axes to plot on.
        figsize: Figure size if creating new figure.
        title: Chart title (auto-generated if None).
        show_scatter: Whether to show individual data points.
        show_legend: Whether to show the legend.
        show_ci: Whether to show error bars (using logged std values).
        x_percentile_range: Percentile range for x-axis limits, e.g. (1, 99).
        colors: Optional list of colors for each mean_key. Defaults to TERM_COLORS.

    Returns:
        The matplotlib Figure object.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()
    # normalize mean_keys to list
    if isinstance(mean_keys, str):
        mean_keys = [mean_keys]
    else:
        mean_keys = list(mean_keys)
    # filter to keys that exist in history_df
    valid_mean_keys = [k for k in mean_keys if k in history_df.columns]
    if not valid_mean_keys:
        ax.text(0.5, 0.5, f"None of {mean_keys} found in history", ha="center", va="center", transform=ax.transAxes)
        return fig
    # set up colors
    if colors is None:
        colors = TERM_COLORS
    # compute global x-axis limits from all keys
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for mean_key in valid_mean_keys:
            valid_df = history_df[[step_key, mean_key]].dropna()
            if len(valid_df) > 0:
                all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each mean_key
    for key_idx, mean_key in enumerate(valid_mean_keys):
        color = colors[key_idx % len(colors)]
        std_key = _get_std_key_for_mean_key(mean_key)
        count_key = _get_count_key_for_mean_key(mean_key)
        has_std = std_key is not None and std_key in history_df.columns
        has_count = count_key is not None and count_key in history_df.columns
        # build list of columns to select
        cols_to_select = [step_key, mean_key]
        if has_std:
            cols_to_select.append(std_key)
        if has_count:
            cols_to_select.append(count_key)
        valid_df = history_df[cols_to_select].dropna(subset=[step_key, mean_key]).copy()
        if len(valid_df) == 0:
            continue
        valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
        x_vals = valid_df[step_key].values
        mean_vals = valid_df[mean_key].values
        # extract short label (train or eval)
        label_prefix = "train" if "train/" in mean_key else ("eval" if "eval/" in mean_key else mean_key)
        # plot error bars using logged std
        if show_ci and has_std:
            std_vals = valid_df[std_key].fillna(0).values
            if has_count:
                counts = valid_df[count_key].fillna(1).values
                counts = np.maximum(counts, 1)
                standard_error = std_vals / np.sqrt(counts)
                z_score_95_ci = 1.96
                lower_ci = mean_vals - z_score_95_ci * standard_error
                upper_ci = mean_vals + z_score_95_ci * standard_error
            else:
                lower_ci = mean_vals - std_vals
                upper_ci = mean_vals + std_vals
            ax.fill_between(x_vals, lower_ci, upper_ci, alpha=0.2, color=color)
        # plot scatter points (mean values)
        if show_scatter:
            ax.scatter(x_vals, mean_vals, alpha=0.4, s=12, color=color)
        # plot line connecting mean values
        ax.plot(x_vals, mean_vals, color=color, linewidth=1.5, alpha=0.8, label=f"{label_prefix} (n={len(valid_df)})")
    # configure axes
    ax.set_xlabel(step_key)
    ax.set_ylabel("reward")
    ax.set_title(title or "Total Reward Over Training")
    if x_limits is not None:
        ax.set_xlim(*x_limits)
    ax.grid(axis="y", alpha=0.3)
    if show_legend:
        ax.legend(loc="upper left", fontsize=8)
    return fig


fig = plot_reward_over_time(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

In [ ]:
def _extract_term_name(key: str) -> str:
    """Extract the term/metric name from a full key path."""
    # remove /mean, /std, /count suffixes
    base = key
    for suffix in ["/mean", "/std", "/count", "/min", "/max"]:
        if base.endswith(suffix):
            base = base[: -len(suffix)]
            break
    # get the last part (e.g., 'soft_match' from 'train/reward/run/terms/soft_match')
    return base.split("/")[-1]


def _group_keys_by_term(
    keys: list[str],
    filter_suffix: str = "/mean",
) -> dict[str, dict[str, str]]:
    """Group keys by term name, separating train and eval.

    Returns:
        Dict mapping term_name -> {"train": train_key, "eval": eval_key}
    """
    grouped: dict[str, dict[str, str]] = {}
    for key in keys:
        if not key.endswith(filter_suffix):
            continue
        term_name = _extract_term_name(key)
        if term_name not in grouped:
            grouped[term_name] = {}
        if "train/" in key:
            grouped[term_name]["train"] = key
        elif "eval/" in key:
            grouped[term_name]["eval"] = key
    return grouped


def plot_reward_terms_over_time(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    figsize_per_subplot: tuple[int, int] = (7, 4),
    max_cols: int = 2,
    show_ci: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot reward terms over time with train/eval on the same subplot for each term.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize_per_subplot: Size per subplot (width, height).
        max_cols: Maximum number of columns in subplot grid.
        show_ci: Whether to show error bands (using logged std values).
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    # find all reward term keys and group by term name
    term_keys = [k for k in history_df.columns if "/reward/run/terms/" in k]
    grouped = _group_keys_by_term(term_keys, filter_suffix="/mean")
    if not grouped:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, "No reward term keys found", ha="center", va="center", transform=ax.transAxes)
        return fig
    # set up subplot grid
    n_terms = len(grouped)
    n_cols = min(max_cols, n_terms)
    n_rows = (n_terms + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_subplot[0] * n_cols, figsize_per_subplot[1] * n_rows))
    if n_terms == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    # compute global x-axis limits
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for term_keys_dict in grouped.values():
            for mean_key in term_keys_dict.values():
                if mean_key in history_df.columns:
                    valid_df = history_df[[step_key, mean_key]].dropna()
                    if len(valid_df) > 0:
                        all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each term
    split_colors = {"train": TERM_COLORS[0], "eval": TERM_COLORS[1]}
    for ax_idx, (term_name, keys_dict) in enumerate(sorted(grouped.items())):
        ax = axes[ax_idx]
        for split, mean_key in keys_dict.items():
            if mean_key not in history_df.columns:
                continue
            color = split_colors[split]
            std_key = _get_std_key_for_mean_key(mean_key)
            count_key = _get_count_key_for_mean_key(mean_key)
            has_std = std_key is not None and std_key in history_df.columns
            has_count = count_key is not None and count_key in history_df.columns
            cols_to_select = [step_key, mean_key]
            if has_std:
                cols_to_select.append(std_key)
            if has_count:
                cols_to_select.append(count_key)
            valid_df = history_df[cols_to_select].dropna(subset=[step_key, mean_key]).copy()
            if len(valid_df) == 0:
                continue
            valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
            x_vals = valid_df[step_key].values
            mean_vals = valid_df[mean_key].values
            # plot CI band
            if show_ci and has_std:
                std_vals = valid_df[std_key].fillna(0).values
                if has_count:
                    counts = np.maximum(valid_df[count_key].fillna(1).values, 1)
                    standard_error = std_vals / np.sqrt(counts)
                    lower_ci = mean_vals - 1.96 * standard_error
                    upper_ci = mean_vals + 1.96 * standard_error
                else:
                    lower_ci = mean_vals - std_vals
                    upper_ci = mean_vals + std_vals
                ax.fill_between(x_vals, lower_ci, upper_ci, alpha=0.2, color=color)
            # plot line and scatter
            ax.scatter(x_vals, mean_vals, alpha=0.4, s=12, color=color)
            ax.plot(x_vals, mean_vals, color=color, linewidth=1.5, alpha=0.8, label=f"{split} (n={len(valid_df)})")
        ax.set_xlabel(step_key)
        ax.set_ylabel("reward")
        ax.set_title(term_name)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)
        if show_legend:
            ax.legend(loc="best", fontsize=8)
    # hide unused subplots
    for ax_idx in range(len(grouped), len(axes)):
        axes[ax_idx].set_visible(False)
    return fig


fig = plot_reward_terms_over_time(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

## Parsing Statistics

In [ ]:
def _extract_parsing_metric_name(key: str) -> str:
    """Extract the metric name from a parsing key path."""
    # e.g., 'train/parsing/missing_answer_ratio' -> 'missing_answer_ratio'
    # e.g., 'train/parsing/output_length_chars/mean' -> 'output_length_chars'
    parts = key.split("/parsing/")
    if len(parts) < 2:
        return key.split("/")[-1]
    metric_part = parts[1]
    # remove /mean suffix if present
    if metric_part.endswith("/mean"):
        metric_part = metric_part[:-5]
    return metric_part


def _group_parsing_keys(
    keys: list[str],
) -> dict[str, dict[str, str]]:
    """Group parsing keys by metric name, separating train and eval.

    Returns:
        Dict mapping metric_name -> {"train": train_key, "eval": eval_key}
    """
    grouped: dict[str, dict[str, str]] = {}
    for key in keys:
        metric_name = _extract_parsing_metric_name(key)
        if metric_name not in grouped:
            grouped[metric_name] = {}
        if "train/" in key:
            grouped[metric_name]["train"] = key
        elif "eval/" in key:
            grouped[metric_name]["eval"] = key
    return grouped


def plot_parsing_stats(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    figsize_per_subplot: tuple[int, int] = (7, 4),
    max_cols: int = 2,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot parsing metrics over time with train/eval on the same subplot for each metric.

    Note: Parsing metrics don't have logged std values, so no CI bands are shown.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize_per_subplot: Size per subplot (width, height).
        max_cols: Maximum number of columns in subplot grid.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    # find all parsing keys and group by metric name
    parsing_keys = [k for k in history_df.columns if "/parsing/" in k]
    grouped = _group_parsing_keys(parsing_keys)
    if not grouped:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, "No parsing metrics found", ha="center", va="center", transform=ax.transAxes)
        return fig
    # set up subplot grid
    n_metrics = len(grouped)
    n_cols = min(max_cols, n_metrics)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_subplot[0] * n_cols, figsize_per_subplot[1] * n_rows))
    if n_metrics == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    # compute global x-axis limits
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for keys_dict in grouped.values():
            for key in keys_dict.values():
                if key in history_df.columns:
                    valid_df = history_df[[step_key, key]].dropna()
                    if len(valid_df) > 0:
                        all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each metric
    split_colors = {"train": TERM_COLORS[0], "eval": TERM_COLORS[1]}
    for ax_idx, (metric_name, keys_dict) in enumerate(sorted(grouped.items())):
        ax = axes[ax_idx]
        for split, key in keys_dict.items():
            if key not in history_df.columns:
                continue
            color = split_colors[split]
            valid_df = history_df[[step_key, key]].dropna().copy()
            if len(valid_df) == 0:
                continue
            valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
            x_vals = valid_df[step_key].values
            y_vals = valid_df[key].values
            # plot line and scatter
            ax.scatter(x_vals, y_vals, alpha=0.4, s=12, color=color)
            ax.plot(x_vals, y_vals, color=color, linewidth=1.5, alpha=0.8, label=f"{split} (n={len(valid_df)})")
        ax.set_xlabel(step_key)
        # determine ylabel based on metric type
        if "ratio" in metric_name or "missing" in metric_name:
            ax.set_ylabel("ratio")
        elif "length" in metric_name:
            ax.set_ylabel("length")
        else:
            ax.set_ylabel("value")
        ax.set_title(metric_name)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)
        if show_legend:
            ax.legend(loc="best", fontsize=8)
    # hide unused subplots
    for ax_idx in range(len(grouped), len(axes)):
        axes[ax_idx].set_visible(False)
    return fig


fig = plot_parsing_stats(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

## Sample Output Viewer

In [ ]:
def normalize_rewards_table(
    df: pd.DataFrame,
    use_logged_step_as_step: bool = False,
) -> pd.DataFrame:
    """Normalize rewards table to consistent schema.

    Expected columns from WandBRewardLogger (see pyine/organisms/models/rewards/core/logging.py):
        sample_id, generation_count, step, prompt, expected_output, model_output,
        reasoning, final_answer, reward_total, reward_terms_json, reward_terms_raw_json,
        reward_metrics_json, categories_json, tags_json, generation_idx

    Args:
        df: Raw rewards table DataFrame.
        use_logged_step_as_step: If True, use _logged_step (the W&B step when the table was
            flushed) as the authoritative step value instead of the logged 'step' column.
            Defaults to False (use the logged 'step' value).

    Returns:
        Normalized DataFrame with parsed JSON columns and consistent schema.
    """
    df = df.copy()
    # parse JSON columns
    json_columns = [
        "reward_terms_json",
        "reward_terms_raw_json",
        "reward_metrics_json",
        "categories_json",
        "tags_json",
    ]
    for col in json_columns:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    # ensure all expected columns exist (matches WandBRewardLogger schema)
    expected_cols = [
        "sample_id",
        "generation_count",
        "step",
        "prompt",
        "expected_output",
        "model_output",
        "reasoning",
        "final_answer",
        "reward_total",
        "reward_terms_json",
        "reward_terms_raw_json",
        "reward_metrics_json",
        "categories_json",
        "tags_json",
        "generation_idx",
    ]
    for col in expected_cols:
        if col not in df.columns:
            df[col] = None
    # optionally use _logged_step as the authoritative step (added by fetch_tables_with_steps)
    if use_logged_step_as_step and "_logged_step" in df.columns:
        df["step"] = df["_logged_step"].fillna(df["step"])
    # sort by step numerically (ensure numeric type for proper sorting)
    if "step" in df.columns:
        df["step"] = pd.to_numeric(df["step"], errors="coerce")
        df = df.sort_values("step", na_position="last").reset_index(drop=True)
    return df


# fetch tables with their true logged steps from W&B
rewards_table = wandb_utils.fetch_tables_with_steps(run, "train/generation_details")
if rewards_table is not None:
    rewards_table = normalize_rewards_table(rewards_table)
    print(f"Loaded {len(rewards_table)} sample records from rewards table")
    print(f"Columns: {list(rewards_table.columns)}")
    if "_logged_step" in rewards_table.columns:
        unique_steps = rewards_table["_logged_step"].nunique()
        print(f"Samples from {unique_steps} unique logged steps")
else:
    print("No rewards table found (log_tables may be disabled in reward logging config)")

In [ ]:
def get_parsing_metric(row: pd.Series, key: str, default: int = 0) -> int:
    """Get a parsing metric from reward_metrics_json (handles prefixed keys)."""
    metrics = row.get("reward_metrics_json")
    if isinstance(metrics, dict):
        # try exact key first, then search for keys ending with the expected suffix
        if key in metrics:
            return metrics[key]
        suffix = f"parsing/{key}"
        for k, v in metrics.items():
            if k.endswith(suffix):
                return v
    return default


def display_sample(row: pd.Series) -> None:
    """Display a single sample from the rewards table."""
    # check for missing fields using actual values
    reasoning = row.get("reasoning")
    has_reasoning = bool(reasoning) if isinstance(reasoning, str) else False
    has_answer = get_parsing_metric(row, "has_answer", default=1) == 1
    # build status indicators
    status_parts = []
    if not has_reasoning:
        status_parts.append('<span style="color: orange; font-weight: bold;">⚠ NO REASONING</span>')
    if not has_answer:
        status_parts.append('<span style="color: orange; font-weight: bold;">⚠ NO ANSWER</span>')
    status_html = " ".join(status_parts) if status_parts else '<span style="color: green;">✓ OK</span>'
    html_parts = []
    # header with sample_id and status
    sample_id = row.get("sample_id", "N/A")
    html_parts.append(f"<h4>Sample: {sample_id} {status_html}</h4>")
    # metadata table
    html_parts.append("<table style='border-collapse: collapse; margin-bottom: 10px;'>")
    # step info
    step = row.get("step")
    logged_step = row.get("_logged_step")
    step_str = f"{int(step)}" if pd.notna(step) else "N/A"
    if pd.notna(logged_step) and logged_step != step:
        step_str += f" (logged at W&B step {int(logged_step)})"
    html_parts.append(f"<tr><td><b>Step:</b></td><td>{step_str}</td></tr>")
    # generation info
    gen_idx = row.get("generation_idx")
    gen_count = row.get("generation_count")
    if pd.notna(gen_idx) or pd.notna(gen_count):
        gen_str_parts = []
        if pd.notna(gen_idx):
            gen_str_parts.append(f"idx={int(gen_idx)}")
        if pd.notna(gen_count):
            gen_str_parts.append(f"total={int(gen_count)}")
        html_parts.append(f"<tr><td><b>Generation:</b></td><td>{', '.join(gen_str_parts)}</td></tr>")
    # reward
    reward_total = row.get("reward_total")
    reward_str = f"{reward_total:.4f}" if pd.notna(reward_total) else "N/A"
    html_parts.append(f"<tr><td><b>Reward:</b></td><td>{reward_str}</td></tr>")
    # categories
    cats = row.get("categories_json", [])
    if cats and isinstance(cats, list):
        html_parts.append(f"<tr><td><b>Categories:</b></td><td>{', '.join(cats)}</td></tr>")
    # tags
    tags = row.get("tags_json", [])
    if tags and isinstance(tags, list):
        html_parts.append(f"<tr><td><b>Tags:</b></td><td>{', '.join(tags)}</td></tr>")
    html_parts.append("</table>")
    # reward terms
    terms = row.get("reward_terms_json", {})
    if terms and isinstance(terms, dict):
        terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in terms.items())
        html_parts.append(f"<b>Reward Terms:</b> {terms_str}<br>")
    # raw reward terms (if available and different from terms)
    raw_terms = row.get("reward_terms_raw_json", {})
    if raw_terms and isinstance(raw_terms, dict) and raw_terms != terms:
        raw_terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in raw_terms.items())
        html_parts.append(f"<b>Raw Terms:</b> {raw_terms_str}<br>")
    html_parts.append("<hr>")
    # prompt (escape HTML to show raw tags like <final>)
    pre_style = "white-space: pre-wrap; max-height: 300px; overflow-y: auto;"
    prompt = row.get("prompt") or ""
    html_parts.append(f"<b>Prompt:</b><br><pre style='{pre_style}'>{html.escape(prompt)}</pre>")
    # expected output
    expected = row.get("expected_output")
    if expected:
        html_parts.append(
            f"<b>Expected Output:</b><br><pre style='white-space: pre-wrap;'>{html.escape(expected)}</pre>"
        )
    # model output (escape HTML to show raw tags like <final>)
    pre_style_tall = "white-space: pre-wrap; max-height: 400px; overflow-y: auto;"
    output = row.get("model_output") or ""
    html_parts.append(f"<b>Model Output:</b><br><pre style='{pre_style_tall}'>{html.escape(output)}</pre>")
    # reasoning (only show separately if different from model_output)
    if has_reasoning and reasoning != output:
        html_parts.append(f"<b>Reasoning:</b><br><pre style='{pre_style}'>{html.escape(reasoning)}</pre>")
    elif not has_reasoning:
        html_parts.append('<b>Reasoning:</b> <span style="color: orange;">(missing)</span><br>')
    # final answer
    answer = row.get("final_answer")
    if answer:
        html_parts.append(f"<b>Final Answer:</b><br><pre style='white-space: pre-wrap;'>{html.escape(answer)}</pre>")
    elif not has_answer:
        html_parts.append('<b>Final Answer:</b> <span style="color: orange;">(missing)</span><br>')
    ipy_display.display(ipy_display.HTML("".join(html_parts)))


if rewards_table is not None and len(rewards_table) > 0:
    print(f"Next cell can display up to {len(rewards_table)} samples.")
else:
    print("No samples to display")

In [ ]:
# interactive sample browser using ipywidgets
try:
    import ipywidgets

    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available - interactive browser disabled")


def create_sample_browser(
    rewards_table: pd.DataFrame,
    run_url: str,
) -> "ipywidgets.VBox":
    """Create an interactive sample browser widget."""
    current_df = rewards_table.copy()
    # filter controls
    show_missing_answer = ipywidgets.Checkbox(value=False, description="Missing answer")
    show_missing_reasoning = ipywidgets.Checkbox(value=False, description="Missing reasoning")
    # navigation
    idx_slider = ipywidgets.IntSlider(value=0, min=0, max=max(0, len(current_df) - 1), description="Index:")
    prev_btn = ipywidgets.Button(description="< Prev")
    next_btn = ipywidgets.Button(description="Next >")
    # output area
    output = ipywidgets.Output()
    info_label = ipywidgets.HTML(value=f"Total samples: {len(current_df)}")

    def apply_filters() -> None:
        nonlocal current_df
        df = rewards_table.copy()
        if show_missing_answer.value:
            df = df[df.apply(lambda r: get_parsing_metric(r, "has_answer", default=1) == 0, axis=1)]
        if show_missing_reasoning.value:
            df = df[df["reasoning"].apply(lambda x: not x if isinstance(x, str) else True)]
        current_df = df
        idx_slider.max = max(0, len(current_df) - 1)
        idx_slider.value = min(idx_slider.value, idx_slider.max)
        info_label.value = f"Showing {len(current_df)} of {len(rewards_table)} samples"
        update_display(None)

    def update_display(_: typing.Any) -> None:
        output.clear_output()
        with output:
            if len(current_df) == 0:
                print("No samples match current filters")
                return
            idx = idx_slider.value
            if idx < len(current_df):
                display_sample(current_df.iloc[idx])

    def on_prev(_: typing.Any) -> None:
        if idx_slider.value > 0:
            idx_slider.value -= 1

    def on_next(_: typing.Any) -> None:
        if idx_slider.value < idx_slider.max:
            idx_slider.value += 1

    # connect handlers
    idx_slider.observe(update_display, names="value")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    show_missing_answer.observe(lambda _: apply_filters(), names="value")
    show_missing_reasoning.observe(lambda _: apply_filters(), names="value")
    # initial display
    update_display(None)
    # layout
    filter_box = ipywidgets.HBox([show_missing_answer, show_missing_reasoning])
    nav_box = ipywidgets.HBox([prev_btn, idx_slider, next_btn])
    url_html = ipywidgets.HTML(value=f"<a href='{run_url}' target='_blank'>Open in W&B</a>")
    return ipywidgets.VBox([info_label, filter_box, nav_box, url_html, output])


if WIDGETS_AVAILABLE and rewards_table is not None and len(rewards_table) > 0:
    browser = create_sample_browser(rewards_table, run.url)
    ipy_display.display(browser)
elif rewards_table is None:
    print("No rewards table available for browsing")
else:
    print("No samples in rewards table")

In [ ]:
# sample filtering configuration
FILTER_STEP_RANGE = (None, None)  # (min_step, max_step) or None
FILTER_REWARD_RANGE = (None, None)  # (min_reward, max_reward) or None
FILTER_CATEGORIES: list[str] = []  # list of category strings to include (empty = all)


def filter_samples(
    rewards_table: pd.DataFrame,
    step_range: tuple[int | None, int | None] = (None, None),
    reward_range: tuple[float | None, float | None] = (None, None),
    categories: list[str] | None = None,
) -> pd.DataFrame:
    """Filter samples by step, reward, or category."""
    df = rewards_table.copy()
    # step range filter
    if step_range[0] is not None and "step" in df.columns:
        df = df[df["step"] >= step_range[0]]
    if step_range[1] is not None and "step" in df.columns:
        df = df[df["step"] <= step_range[1]]
    # reward range filter
    if reward_range[0] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] >= reward_range[0]]
    if reward_range[1] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] <= reward_range[1]]
    # category filter
    if categories and "categories_json" in df.columns:

        def has_category(cats: typing.Any) -> bool:
            if not isinstance(cats, list):
                return False
            return any(c in cats for c in categories)

        df = df[df["categories_json"].apply(has_category)]
    return df


if rewards_table is not None:
    filtered = filter_samples(rewards_table, FILTER_STEP_RANGE, FILTER_REWARD_RANGE, FILTER_CATEGORIES)
    print(f"Filtered to {len(filtered)} samples (from {len(rewards_table)} total)")
else:
    filtered = None
    print("No rewards table to filter")

In [ ]:
def plot_reward_distributions(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 10),
) -> matplotlib.figure.Figure:
    """Plot reward distributions."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    # 1. total reward histogram
    ax = axes[0, 0]
    if "reward_total" in rewards_table.columns:
        rewards = rewards_table["reward_total"].dropna()
        ax.hist(rewards, bins=50, color="#2C7BB6", alpha=0.7, edgecolor="black")
        ax.axvline(rewards.mean(), color="red", linestyle="--", label=f"Mean: {rewards.mean():.3f}")
        ax.set_xlabel("Reward")
        ax.set_ylabel("Count")
        ax.legend()
    ax.set_title("Total Reward Distribution")
    ax.grid(True, alpha=0.3)
    # 2. reward over steps
    ax = axes[0, 1]
    if "step" in rewards_table.columns and "reward_total" in rewards_table.columns:
        ax.scatter(rewards_table["step"], rewards_table["reward_total"], alpha=0.3, s=10)
    ax.set_title("Reward vs Step")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    # 3. reward terms breakdown (if available)
    ax = axes[1, 0]
    if "reward_terms_json" in rewards_table.columns:
        # extract term values
        term_data: dict[str, list[float]] = {}
        for terms in rewards_table["reward_terms_json"].dropna():
            if isinstance(terms, dict):
                for term, value in terms.items():
                    if term not in term_data:
                        term_data[term] = []
                    term_data[term].append(value)
        if term_data:
            term_names = list(term_data.keys())
            term_values = [term_data[t] for t in term_names]
            ax.boxplot(term_values, tick_labels=term_names)
            ax.set_xticklabels(term_names, rotation=45, ha="right")
    ax.set_title("Reward Terms Distribution")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    # 4. rewards by category (if available)
    ax = axes[1, 1]
    if "categories_json" in rewards_table.columns and "reward_total" in rewards_table.columns:
        # group by first category
        cat_rewards: dict[str, list[float]] = {}
        for _idx, row in rewards_table.iterrows():
            cats = row.get("categories_json", [])
            reward = row.get("reward_total")
            if isinstance(cats, list) and cats and reward is not None:
                cat = cats[0]  # use first category
                if cat not in cat_rewards:
                    cat_rewards[cat] = []
                cat_rewards[cat].append(reward)
        if cat_rewards:
            cat_names = list(cat_rewards.keys())[:8]  # limit to 8
            cat_values = [cat_rewards[c] for c in cat_names]
            ax.boxplot(cat_values, tick_labels=cat_names)
            ax.set_xticklabels(cat_names, rotation=45, ha="right", fontsize=8)
    ax.set_title("Rewards by Category")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_reward_distributions(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for distribution analysis")

In [ ]:
def plot_length_vs_reward(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 5),
) -> matplotlib.figure.Figure:
    """Scatter plot: output/reasoning length vs reward."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    # compute lengths if not already present
    df = rewards_table.copy()
    if "output_length" not in df.columns and "model_output" in df.columns:
        df["output_length"] = df["model_output"].apply(lambda x: len(str(x)) if x else 0)
    if "reasoning_length" not in df.columns and "reasoning" in df.columns:
        df["reasoning_length"] = df["reasoning"].apply(lambda x: len(str(x)) if x else 0)
    # output length vs reward
    ax = axes[0]
    if "output_length" in df.columns and "reward_total" in df.columns:
        valid = df[["output_length", "reward_total"]].dropna()
        ax.scatter(valid["output_length"], valid["reward_total"], alpha=0.3, s=10, c="#2C7BB6")
        # add trend line
        if len(valid) > 10:
            z = np.polyfit(valid["output_length"], valid["reward_total"], 1)
            p = np.poly1d(z)
            x_line = np.linspace(valid["output_length"].min(), valid["output_length"].max(), 100)
            ax.plot(x_line, p(x_line), "r--", alpha=0.7, label="Trend")
            ax.legend()
    ax.set_xlabel("Output Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Output Length vs Reward")
    ax.grid(True, alpha=0.3)
    # reasoning length vs reward
    ax = axes[1]
    if "reasoning_length" in df.columns and "reward_total" in df.columns:
        valid = df[["reasoning_length", "reward_total"]].dropna()
        valid = valid[valid["reasoning_length"] > 0]  # only samples with reasoning
        if len(valid) > 0:
            ax.scatter(valid["reasoning_length"], valid["reward_total"], alpha=0.3, s=10, c="#7570B3")
            if len(valid) > 10:
                z = np.polyfit(valid["reasoning_length"], valid["reward_total"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(valid["reasoning_length"].min(), valid["reasoning_length"].max(), 100)
                ax.plot(x_line, p(x_line), "r--", alpha=0.7, label="Trend")
                ax.legend()
        else:
            ax.text(
                0.5,
                0.5,
                "No samples with reasoning",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
    ax.set_xlabel("Reasoning Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Reasoning Length vs Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_length_vs_reward(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for length analysis")

## TRL Metrics

In [ ]:
def plot_trl_metrics(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    figsize: tuple[int, int] = (14, 5),
    show_scatter: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot TRL-related metrics: entropy, KL divergence, and clip ratios.

    Shows train/eval curves together where available. Metrics without data show a placeholder.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize: Figure size.
        show_scatter: Whether to show individual data points.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    split_colors = {"train": TERM_COLORS[0], "eval": TERM_COLORS[1]}
    # define metrics to plot: (title, ylabel, key_patterns)
    metrics_config = [
        ("Entropy", "entropy", ["train/entropy", "eval/entropy"]),
        ("KL Divergence", "kl", ["train/kl", "eval/kl", "kl"]),
        ("Clip Ratio", "clip_ratio", [k for k in history_df.columns if "clip_ratio" in k]),
    ]
    # compute global x-axis limits
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for _, _, keys in metrics_config:
            for key in keys:
                if key in history_df.columns:
                    valid_df = history_df[[step_key, key]].dropna()
                    if len(valid_df) > 0:
                        all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each metric
    for ax_idx, (title, ylabel, candidate_keys) in enumerate(metrics_config):
        ax = axes[ax_idx]
        # filter to keys that exist in history_df
        valid_keys = [k for k in candidate_keys if k in history_df.columns]
        if not valid_keys:
            ax.text(0.5, 0.5, f"No {ylabel} metrics found", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)
            ax.set_xlabel(step_key)
            ax.set_ylabel(ylabel)
            ax.grid(axis="y", alpha=0.3)
            continue
        plotted_any = False
        for key in valid_keys:
            valid_df = history_df[[step_key, key]].dropna().copy()
            if len(valid_df) == 0:
                continue
            valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
            x_vals = valid_df[step_key].values
            y_vals = valid_df[key].values
            # determine color based on split
            if "train/" in key:
                color = split_colors["train"]
                label = f"train (n={len(valid_df)})"
            elif "eval/" in key:
                color = split_colors["eval"]
                label = f"eval (n={len(valid_df)})"
            else:
                color = TERM_COLORS[2]
                short_key = key.split("/")[-1] if "/" in key else key
                label = f"{short_key} (n={len(valid_df)})"
            ax.plot(x_vals, y_vals, color=color, linewidth=1.5, alpha=0.8)
            if show_scatter:
                ax.scatter(x_vals, y_vals, color=color, s=12, alpha=0.4, label=label)
            else:
                ax.plot([], [], color=color, label=label)  # for legend
            plotted_any = True
        if not plotted_any:
            ax.text(0.5, 0.5, f"{ylabel} keys found but no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(step_key)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)
        if show_legend and plotted_any:
            ax.legend(loc="best", fontsize=8)
    return fig


fig = plot_trl_metrics(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

# print data availability summary
print("\nTRL Metrics Data Availability:")
trl_keys = [k for k in history_df.columns if any(m in k for m in ["entropy", "kl", "clip_ratio"])]
for key in sorted(trl_keys):
    valid = history_df[key].dropna()
    if len(valid) > 0:
        steps = history_df.loc[valid.index, step_key]
        print(f"  {key}: {len(valid)} samples, steps {steps.min():.0f}-{steps.max():.0f}")
    else:
        print(f"  {key}: column exists but no data")
if not trl_keys:
    print("  No TRL metrics (entropy, kl, clip_ratio) found in history")

## Training Speed & Other Metrics

In [ ]:
def plot_training_speed_metrics(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    time_key: str | None = None,
    figsize: tuple[int, int] = (14, 10),
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot training speed and throughput metrics.

    Shows train/eval curves together where available.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        time_key: Name of the time column (_timestamp or _runtime). If None, auto-detected.
        figsize: Figure size.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    axes = axes.flatten()
    split_colors = {"train": TERM_COLORS[0], "eval": TERM_COLORS[1]}
    # determine time key if not provided
    if time_key is None:
        time_key = "_runtime" if "_runtime" in history_df.columns else "_timestamp"
    # define metrics to plot: (title, ylabel, train_key, eval_key)
    metrics_config = [
        ("Failure Counts", "count", "train/failures/failure_count", "eval/failures/failure_count"),
        ("Step Time", "seconds", "train/step_time", "eval/step_time"),
        ("Samples per Second", "samples/s", "train/samples_per_second", "eval/samples_per_second"),
        ("Steps per Second", "steps/s", "train/steps_per_second", "eval/steps_per_second"),
        ("Token Count", "tokens", "train/num_tokens", "eval/num_tokens"),
    ]
    # compute global x-axis limits
    x_limits: tuple[float, float] | None = None
    if x_percentile_range is not None:
        all_x_values = []
        for _, _, train_key, eval_key in metrics_config:
            for key in [train_key, eval_key]:
                if key in history_df.columns:
                    valid_df = history_df[[step_key, key]].dropna()
                    if len(valid_df) > 0:
                        all_x_values.extend(valid_df[step_key].values)
        if all_x_values:
            x_low = float(np.percentile(all_x_values, x_percentile_range[0]))
            x_high = float(np.percentile(all_x_values, x_percentile_range[1]))
            if x_low < x_high:
                padding = (x_high - x_low) * 0.05
                x_limits = (x_low - padding, x_high + padding)
    # plot each metric
    for ax_idx, (title, ylabel, train_key, eval_key) in enumerate(metrics_config):
        ax = axes[ax_idx]
        plotted_any = False
        for key, split in [(train_key, "train"), (eval_key, "eval")]:
            if key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna().copy()
            if len(valid_df) == 0:
                continue
            valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
            x_vals = valid_df[step_key].values
            y_vals = valid_df[key].values
            color = split_colors[split]
            ax.plot(x_vals, y_vals, color=color, linewidth=1.5, alpha=0.8)
            ax.scatter(x_vals, y_vals, color=color, s=12, alpha=0.4, label=f"{split} (n={len(valid_df)})")
            plotted_any = True
        if not plotted_any:
            ax.text(0.5, 0.5, f"No {ylabel} metrics found", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(step_key)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)
        if show_legend and plotted_any:
            ax.legend(loc="best", fontsize=8)
    # last subplot: training progress (steps vs wall-clock time)
    ax = axes[5]
    if time_key in history_df.columns:
        valid_df = history_df[[step_key, time_key]].dropna().sort_values(time_key)
        if len(valid_df) > 0:
            time_vals = valid_df[time_key].values
            step_vals = valid_df[step_key].values
            # convert timestamp to relative time if needed
            if time_key == "_timestamp":
                time_vals = time_vals - time_vals[0]
                x_label = "Time (seconds since start)"
            else:
                x_label = "Runtime (seconds)"
            # convert to minutes if > 60 seconds
            if time_vals.max() > 60:
                time_vals = time_vals / 60
                x_label = x_label.replace("seconds", "minutes")
            # convert to hours if > 60 minutes
            if time_vals.max() > 60:
                time_vals = time_vals / 60
                x_label = x_label.replace("minutes", "hours")
            ax.plot(time_vals, step_vals, color=TERM_COLORS[0], linewidth=2)
            ax.scatter(
                time_vals[:: max(1, len(time_vals) // 50)],
                step_vals[:: max(1, len(step_vals) // 50)],
                color=TERM_COLORS[0],
                alpha=0.5,
                s=20,
            )
            # compute and show average speed
            if len(time_vals) > 1 and time_vals[-1] > time_vals[0]:
                total_steps = step_vals[-1] - step_vals[0]
                total_time = time_vals[-1] - time_vals[0]
                avg_speed = total_steps / total_time
                unit = x_label.split("(")[-1].split(")")[0].split()[-1]
                ax.text(
                    0.02,
                    0.98,
                    f"Avg: {avg_speed:.2f} steps/{unit}",
                    transform=ax.transAxes,
                    fontsize=9,
                    verticalalignment="top",
                    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
                )
            ax.set_xlabel(x_label)
            ax.set_ylabel(step_key)
            ax.set_title("Training Progress (Steps vs Wall-Clock)")
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, "No time data available", ha="center", va="center", transform=ax.transAxes)
            ax.set_title("Training Progress")
    else:
        ax.text(0.5, 0.5, "No time key available", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Training Progress")
    return fig


fig = plot_training_speed_metrics(history_df, step_key=step_key, time_key=time_key)
plt.tight_layout()
plt.show()

# print summary of available metrics
print("\nTraining Speed Metrics Summary:")
speed_keys = [
    k for k in history_df.columns if any(s in k for s in ["failure", "step_time", "per_second", "num_tokens"])
]
for key in sorted(speed_keys):
    if key in history_df.columns:
        valid = history_df[key].dropna()
        if len(valid) > 0:
            print(f"  {key}: n={len(valid)}, mean={valid.mean():.4f}, std={valid.std():.4f}")
        else:
            print(f"  {key}: no data")

# print time-based stats if available
if time_key and time_key in history_df.columns:
    valid_time = history_df[[step_key, time_key]].dropna()
    if len(valid_time) > 1:
        total_time_seconds = valid_time[time_key].max() - valid_time[time_key].min()
        total_steps = valid_time[step_key].max() - valid_time[step_key].min()
        print("\nExperiment Duration:")
        if total_time_seconds > 3600:
            print(f"  Total time: {total_time_seconds / 3600:.2f} hours")
        elif total_time_seconds > 60:
            print(f"  Total time: {total_time_seconds / 60:.2f} minutes")
        else:
            print(f"  Total time: {total_time_seconds:.2f} seconds")
        print(f"  Total steps: {total_steps:.0f}")
        print(f"  Average speed: {total_steps / total_time_seconds:.2f} steps/second")

# print generation statistics
print("\nGeneration Statistics:")
for split in ["train", "eval"]:
    gen_key = f"{split}/generation_count"
    if gen_key in history_df.columns:
        valid = history_df[gen_key].dropna()
        if len(valid) > 0:
            total_generations = int(valid.sum())
            print(f"  {split} total generations: {total_generations:,}")
            # compute generations per second if time data available
            if time_key and time_key in history_df.columns:
                # get time range for this split's generations
                gen_df = history_df[[gen_key, time_key]].dropna()
                if len(gen_df) > 1:
                    gen_time_seconds = gen_df[time_key].max() - gen_df[time_key].min()
                    if gen_time_seconds > 0:
                        gen_per_second = total_generations / gen_time_seconds
                        print(f"  {split} generations/second: {gen_per_second:.2f}")
        else:
            print(f"  {split} generation_count: no data")
    else:
        print(f"  {split} generation_count: not found")